# Curriculum 04 · Lab 4 — Parent-child retrieval: small chunks in, large context out

**Goal:** Resolve the chunk-size tension. Small chunks match a query
precisely but carry too little context for an answer; large chunks carry
context but match coarsely. Parent-child retrieval splits the difference:
embed SMALL child chunks for precise matching, but return the LARGER parent
document each child came from whenever a child hits.

```
Retriever : ParentDocumentRetriever (langchain-classic)
CHILDREN  : 120-char chunks — embedded and matched against the query
PARENTS   : ~500-char chunks — the contexts returned to the caller
Docstore  : InMemoryStore maps each child's doc_id back to its parent
Store     : FAISS in-memory — nothing written to disk
Embedding : BGE (BAAI/bge-base-en-v1.5, local, CPU)
Data      : rag-mini-wikipedia (first 20 passages -> 80+ children)
```

**Why parent-child:** it is the "small chunks in, large context out" trick
behind Project 09: the retriever's precision comes from the small chunk, the
answer step's context from the large one. The demo prints the child that
matched, the parent returned, and the `x` factor of context you gained.

This is the fourth lab of track 04-retrieval (see
`.omo/plans/layer1-rag-playbook.md`).


## 0 · Setup — environment, imports & repo paths

**WHAT:** Installs the lab's dependencies (a no-op if already present),
imports pandas plus the repo's retriever/store classes and BGE embedder, and
puts the repo-root component library on `sys.path` so this notebook reuses
`retrieval/*.py`, `vectordb/faiss.py` and `embeddings/bge.py` exactly like the
lab script.

**WHY:** Everything embeds **locally** with BGE via sentence-transformers —
no API embeddings anywhere. The retriever classes live in the repo's shared
component library (`retrieval/`), not inside the lab, so the exact same code
path runs here, in the `.py`, and in later tracks.

**Paths:** the next cell resolves the **repo root** automatically — it works
whether the kernel launches from the repo root (like the lab script) or from
the notebook's own folder (the Jupyter default) — and `cd`s into it so every
path stays repo-relative.

**WHAT TO EXPECT:** no output from the pip cell (packages already
installed), a silent import from the second. The BGE model is loaded lazily
when the experiment cell first calls it.


In [1]:
# Lab-specific dependencies (already in requirements.txt — the install is a
# no-op safety net for fresh environments):
#   sentence-transformers -> local BGE embeddings (langchain_huggingface)
#   faiss-cpu             -> the FAISS vector store (langchain_community)
#   langchain-classic     -> ParentDocumentRetriever
#   langchain-text-splitters -> the child/parent RecursiveCharacterTextSplitter
#   pandas                -> reads the passages/test.parquet corpus
%pip install sentence-transformers faiss-cpu langchain-classic langchain-text-splitters pandas



[notice] A new release of pip is available: 24.2 -> 26.2
[notice] To update, run: python -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


In [2]:
from __future__ import annotations

import os
import sys
import time
from pathlib import Path

import pandas as pd

# Make the repo-root component library importable. A notebook has no
# ``__file__``, so resolve the repo root by walking up from the kernel's
# working directory — this works whether the kernel launches from the repo
# root (like the lab script) or from the notebook's own folder (Jupyter's
# default) — then cd into it so every repo-relative path behaves exactly
# like the .py.
REPO_ROOT = Path.cwd()
for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / "curriculum").is_dir() and (candidate / "NoteBooks").is_dir():
        REPO_ROOT = candidate
        break
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT))

from langchain_classic.retrievers import ParentDocumentRetriever  # noqa: E402
from langchain_community.vectorstores import FAISS  # noqa: E402
from langchain_core.documents import Document  # noqa: E402
from langchain_core.stores import InMemoryStore  # noqa: E402
from langchain_huggingface import HuggingFaceEmbeddings  # noqa: E402
from langchain_text_splitters import RecursiveCharacterTextSplitter  # noqa: E402


/tmp/ipykernel_1405908/1570993024.py:25: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS  # noqa: E402


## 1 · Configuration — the experiment's knobs

**WHAT:** The corpus constants (`N_PARENTS = 20` passages as parents) plus
the splitter geometry: `CHILD_CHUNK_SIZE = 120` (embedded and matched),
`CHILD_OVERLAP = 20`, `PARENT_CHUNK_SIZE = 500` (returned to the caller),
`PARENT_OVERLAP = 50`, and `K = 3` (how many parents each query asks for).

**WHY:** The 120/500 split is the whole lab: children are ~4x smaller than
parents, so matching precision and answer context come from different chunk
sizes — and the gap is what the demo quantifies per query.


In [3]:
PASSAGES_PATH = Path("Data/corpus/rag-mini-wikipedia/passages.parquet")
TEST_PATH = Path("Data/corpus/rag-mini-wikipedia/test.parquet")
N_PARENTS = 20  # deterministic head of the 3200-passage corpus (keeps runtime low)
QUESTION_IDS = [1606, 1610, 1604]  # real questions from test.parquet, answers inside the subset
CHILD_CHUNK_SIZE = 120  # small chunks: embedded and matched against the query
CHILD_OVERLAP = 20
PARENT_CHUNK_SIZE = 500  # large contexts: returned to the caller
PARENT_OVERLAP = 50
K = 3  # search_kwargs: how many parents each query returns
BGE_MODEL_NAME = "BAAI/bge-base-en-v1.5"
PREVIEW = 62  # max characters of chunk text shown next to each hit


## 2 · Load — parents + questions from the fresh parquet files

**WHAT:** `load_parents` returns the first `n` passages as parent Documents
with a string `doc_id` in metadata (the key the docstore links children to);
`load_questions` pulls specific test rows; `preview` flattens a chunk for
one-line printing.

**WHY:** The `doc_id` metadata is the load-bearing part of parent-child: the
retriever stores every child's vector *and* its parent's id, then looks the
parent up in the docstore on hit. String ids keep that contract explicit.


In [4]:
def load_parents(path: Path, n: int) -> list[Document]:
    """Return the first ``n`` passages as parent Documents (id -> doc_id)."""
    df = pd.read_parquet(path)
    subset = df.head(n)
    return [
        Document(page_content=text, metadata={"doc_id": str(i)})
        for i, text in enumerate(subset["passage"].tolist())
    ]


def load_questions(path: Path, ids: list[int]) -> list[tuple[int, str]]:
    """Return [(question_id, question_text)] for the requested test rows."""
    df = pd.read_parquet(path)
    rows = df.loc[ids]
    return [(int(idx), row["question"]) for idx, row in rows.iterrows()]


def preview(text: str, limit: int = PREVIEW) -> str:
    """Flatten a chunk for one-line printing."""
    flat = text.replace("\n", " ")
    return flat[:limit] + ("..." if len(flat) > limit else "")


## 3 · Experiment — split into children, embed, index, query

**WHAT:** `run_experiment` splits the 20 parents into ~80 children, embeds
the children with BGE, builds the `ParentDocumentRetriever` (FAISS over the
children + InMemoryStore over the parents), and retrieves top-K parents for
each question — recording, per query, the parent list, the single matched
child, and the child->parent linkage check.

**WHY:** The artifact pair (matched child + returned parent) is what makes
the "small in, large out" claim checkable: the gate verifies the child is a
literal fragment of the parent it came from.


In [5]:
def run_experiment() -> dict:
    parents = load_parents(PASSAGES_PATH, N_PARENTS)
    questions = load_questions(TEST_PATH, QUESTION_IDS)

    # --- Splitters: one for parents (big), one for children (small) ---------
    child_splitter = RecursiveCharacterTextSplitter(
        chunk_size=CHILD_CHUNK_SIZE, chunk_overlap=CHILD_OVERLAP
    )
    parent_splitter = RecursiveCharacterTextSplitter(
        chunk_size=PARENT_CHUNK_SIZE, chunk_overlap=PARENT_OVERLAP
    )

    # --- Local BGE embeddings (never an API model) --------------------------
    embeddings = HuggingFaceEmbeddings(
        model_name=BGE_MODEL_NAME,
        encode_kwargs={"normalize_embeddings": True},
    )

    # --- In-memory stores ---------------------------------------------------
    # FAISS cannot infer the embedding dimension from an empty list, so seed
    # it with one real parent and delete the seed again (index stays empty).
    seed = FAISS.from_documents([parents[0]], embedding=embeddings)
    seed_id = next(iter(seed.index_to_docstore_id.values()))
    seed.delete([seed_id])
    vs = seed  # holds the CHILDREN
    docstore = InMemoryStore()  # holds the PARENTS, keyed by doc_id

    retriever = ParentDocumentRetriever(
        vectorstore=vs,
        docstore=docstore,
        child_splitter=child_splitter,
        parent_splitter=parent_splitter,
        search_kwargs={"k": K},
    )

    # --- Split + embed + index in one call ----------------------------------
    t0 = time.perf_counter()
    retriever.add_documents(parents, add_to_docstore=True)
    ingest_s = time.perf_counter() - t0

    # Children live inside the FAISS vector store, keyed by index position in
    # index_to_docstore_id; every child knows its parent via metadata.doc_id.
    child_ids = list(vs.index_to_docstore_id.values())
    children = vs.get_by_ids(child_ids)
    parent_ids = list(docstore.yield_keys())
    id_to_parent = dict(zip(parent_ids, docstore.mget(parent_ids)))

    # --- Query: invoke() returns PARENTS; a direct vector search shows the
    #     CHILD that matched ------------------------------------------------
    results = []
    for qid, qtext in questions:
        retrieved = retriever.invoke(qtext)  # K parents
        matched_child, score = vs.similarity_search_with_score(qtext, k=1)[0]
        results.append(
            {
                "qid": qid,
                "qtext": qtext,
                "parents": retrieved,
                "child": matched_child,
                "child_score": score,
                "child_parent": id_to_parent.get(matched_child.metadata.get("doc_id")),
            }
        )

    parent_lens = [len(d.page_content) for d in id_to_parent.values()]
    child_lens = [len(c.page_content) for c in children]

    return {
        "n_parents": len(parent_ids),
        "n_children": len(children),
        "children": children,
        "parent_lens": parent_lens,
        "child_lens": child_lens,
        "results": results,
        "ingest_s": ingest_s,
        "n_queries": len(questions),
    }


## 4 · Run — execute the experiment

**WHAT:** Calls `run_experiment()` — one embed, one index build, all
retrievals — and keeps the artifact dict as `exp`.

**WHY:** Everything after this cell (the demo and the verification gate)
reads from this single `exp`, so the printed numbers and the verified
numbers are guaranteed to come from the same run.


In [6]:
exp = run_experiment()


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

## 5 · Demo — read the artifact

**WHAT:** `print_demo` prints the corpus stats (parents -> children counts,
mean parent/child lengths and their ratio), then per question the top-K
PARENTS with their character counts, the matched CHILD with its score, and
the `child -> parent` context-multiplier line.

**WHY:** The character counts tell the story numerically: a ~120-char child
matches the query, and the retriever returns its ~500-char parent — 2.5–4x
the context of the chunk that actually matched. That multiplier is the
entire point of the pattern.


In [7]:
def print_demo(exp: dict) -> None:
    n_parents, n_children = exp["n_parents"], exp["n_children"]
    mean_p = sum(exp["parent_lens"]) / len(exp["parent_lens"])
    mean_c = sum(exp["child_lens"]) / len(exp["child_lens"])

    print("=" * 66)
    print("Lab 04 — Parent-child retrieval: small chunks in, large context out")
    print(f"{BGE_MODEL_NAME} | FAISS (in-memory) | ParentDocumentRetriever")
    print("=" * 66)

    print(f"\n[1] Corpus (deterministic subset, no randomness):")
    print(f"    {n_parents} parents (first {N_PARENTS} passages of 3200, split at {PARENT_CHUNK_SIZE} chars)")
    print(f"    {n_children} children (embedded at {CHILD_CHUNK_SIZE} chars)")
    print(f"    {exp['n_queries']} questions from test.parquet:")
    for r in exp["results"]:
        print(f"      [{r['qid']}] {r['qtext']}")

    print(f"\n[2] Split + embed + index:")
    print(f"    {n_parents} parents -> {n_children} children in {exp['ingest_s']:.2f}s")
    print(f"    mean parent length {mean_p:.0f} chars vs mean child length {mean_c:.0f} chars"
          f" ({mean_p / mean_c:.1f}x)")
    print(f"    children >> parents: {n_children} child vectors indexed, "
          f"{n_parents} parent docs in the docstore")

    print(f"\n[3] Top-{K} per question (each hit is a PARENT, matched via its children):")
    for r in exp["results"]:
        print(f'\n    Q[{r["qid"]}] "{r["qtext"]}"')
        for rank, doc in enumerate(r["parents"], 1):
            print(f"      {rank}. PARENT ({len(doc.page_content)} chars) "
                  f"[{preview(doc.page_content)}]")
        child = r["child"]
        print(f"      matched CHILD ({len(child.page_content)} chars, "
              f"score {r['child_score']:.4f}):")
        print(f"        [{preview(child.page_content)}]")
        cp = r["child_parent"]
        if cp is not None:
            print(f"      child -> parent ({len(cp.page_content)} chars): the returned "
                  f"context is {len(cp.page_content) / max(len(child.page_content), 1):.1f}x "
                  f"the chunk that matched")

    print("\n[4] Takeaway")
    print("    The retriever embeds 120-char children for precise matching, then")
    print("    maps each hit back through metadata.doc_id to its ~500-char parent.")
    print("    Precision comes from the small chunk, context from the large one —")
    print("    the 'small chunks in, large context out' trick of Project 09.")


In [8]:
print_demo(exp)


Lab 04 — Parent-child retrieval: small chunks in, large context out
BAAI/bge-base-en-v1.5 | FAISS (in-memory) | ParentDocumentRetriever

[1] Corpus (deterministic subset, no randomness):
    25 parents (first 20 passages of 3200, split at 500 chars)
    81 children (embedded at 120 chars)
    3 questions from test.parquet:
      [1606] Is Uruguay's capital Montevideo?
      [1610] Who founded Montevideo?
      [1604] Is Uruguay located in the northwesten part of Africa?

[2] Split + embed + index:
    25 parents -> 81 children in 0.61s
    mean parent length 300 chars vs mean child length 104 chars (2.9x)
    children >> parents: 81 child vectors indexed, 25 parent docs in the docstore

[3] Top-3 per question (each hit is a PARENT, matched via its children):

    Q[1606] "Is Uruguay's capital Montevideo?"
      1. PARENT (498 chars) [Uruguay's capital, Montevideo, was founded by the Spanish in t...]
      2. PARENT (250 chars) [Uruguay (official full name in  ; pron.  , Eastern Republi

## 6 · Verification gate — the same checks the .py runs

**WHAT:** Runs the exact `verify_gate`: children outnumber parents, no empty
children, every retrieved doc is a parent (longer than `CHILD_CHUNK_SIZE`),
each question returns 1..K deduped parents, both content checks (Q1610's
Spanish-founder parent, Q1606's Montevideo parent), and the linkage check —
the matched child text is contained verbatim in its docstore parent.

**WHY:** `python 04-parent-child.py --verify` must print 7/7 PASS; this cell
proves the notebook reproduces the verified `.py` exactly.


In [9]:
def verify_gate(exp: dict) -> int:
    checks: list[tuple[str, bool]] = []

    # The child splitter actually split: many small chunks from few parents.
    checks.append((f"children ({exp['n_children']}) outnumber parents ({exp['n_parents']})",
                   exp["n_children"] > exp["n_parents"]))
    checks.append(("no empty child chunks",
                   all(len(c.page_content) > 0 for c in exp["children"])))

    # Every retrieved hit is a PARENT (longer than any child chunk)...
    all_parents = all(
        len(d.page_content) > CHILD_CHUNK_SIZE
        for r in exp["results"] for d in r["parents"]
    )
    checks.append(("every retrieved doc is a parent (len > CHILD_CHUNK_SIZE)",
                   all_parents))

    # ...and each question returns between 1 and K parents. ParentDocumentRetriever
    # dedupes: several top children can belong to the same parent, so the count
    # is at most K but not always exactly K.
    checks.append((f"each question returns 1..{K} parents (deduped)",
                   all(1 <= len(r["parents"]) <= K for r in exp["results"])))

    # Content: Q1610 "Who founded Montevideo?" must retrieve the parent that
    # says the Spanish founded Montevideo.
    q1610_top = exp["results"][1]["parents"][0].page_content.lower()
    checks.append(("Q1610 top-1 parent names the Spanish founder of Montevideo",
                   "spanish" in q1610_top))

    # Q1606 "Is Uruguay's capital Montevideo?" must retrieve a Uruguay parent
    # that mentions Montevideo.
    q1606_top = exp["results"][0]["parents"][0].page_content.lower()
    checks.append(("Q1606 top-1 parent mentions Montevideo", "montevideo" in q1606_top))

    # Linkage: the matched child is literally a fragment of the parent the
    # docstore says it came from (deterministic — the splitter concatenates
    # substrings, it never rewrites text).
    linked = all(
        r["child_parent"] is not None
        and r["child"].page_content.strip() in r["child_parent"].page_content
        for r in exp["results"]
    )
    checks.append(("matched child text is contained in its docstore parent",
                   linked))

    print("verification gate:")
    for label, ok in checks:
        print(f"  [{'PASS' if ok else 'FAIL'}] {label}")
    return 0 if all(ok for _, ok in checks) else 1


In [10]:
verify_gate(exp)


verification gate:
  [PASS] children (81) outnumber parents (25)
  [PASS] no empty child chunks
  [PASS] every retrieved doc is a parent (len > CHILD_CHUNK_SIZE)
  [PASS] each question returns 1..3 parents (deduped)
  [PASS] Q1610 top-1 parent names the Spanish founder of Montevideo
  [PASS] Q1606 top-1 parent mentions Montevideo
  [PASS] matched child text is contained in its docstore parent


0